## Imports and Loading

In [168]:
import numpy as np
import pandas as pd

In [169]:
import re # for regex matching

In [170]:
df = pd.read_csv("./data/raw.csv")

print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded dataset: 103 rows, 11 columns


In [193]:
# Initialize an empty dataframe to store rejected/bad records
# We copy the columns from the main dataframe so they match perfectly
rejected_records = pd.DataFrame(columns=df.columns)

print("Initialized rejected_records accumulator")

Initialized rejected_records accumulator


## Overview

In [171]:
df.head(10)

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,11/22/2024,Blender,Home,3,38,Cash on Delivery,Shipped,114.00
1,101,Customer_101,ORD-35783,7/5/2025,Smartphone,Electronics,2,abd,PayPal,Processing,NaN
2,102,Customer_102,ORD-84355,12/23/2024,Tennis Racket,Sports,1,389.05,PayPal,Delivered,389.05
3,103,Customer_103,ORD-57811,3/19/2025,Science,Books,5,233.92,PayPal,Processing,1169.60
4,104,Customer_104,ORD-93614,10/20/2025,Biography,Books,1,552.51,Cash on Delivery,Processing,552.51
5,105,Customer_105,ORD-22442,11/20/2024,Tennis Racket,Sports,3,122.06,Cash on Delivery,Cancelled,366.18
6,106,Customer_106,ORD-25885,2/2/2025,Blender,Home,NaN,978.63,Bank Transfer,Processing,NaN
7,107,Customer_107,ORD-89122,1/3/2025,Biography,Books,5,587.68,PayPal,Returned,2938.40
8,108,Customer_108,ORD-64400,10/23/2025,Science,Books,1,600.29,Cash on Delivery,Processing,600.29
9,109,Customer_109,ORD-18512,5/3/2025,Tennis Racket,Sports,5,168.34,Credit Card,Shipped,841.70


In [172]:
# Duplicate check
print("(1) Duplicates:", df.duplicated().sum())
display(df[df.duplicated(keep=False)]) # keep=False to also include the first occurrences
df.drop_duplicates(inplace=True)
print("(2) Duplicates:", df.duplicated().sum())

(1) Duplicates: 1


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
46,146,Customer_146,ORD-32755,7/9/2025,Basketball,electronic,2,705.42,Bank Transfer,Processing,1410.84
102,146,Customer_146,ORD-32755,7/9/2025,Basketball,electronic,2,705.42,Bank Transfer,Processing,1410.84


(2) Duplicates: 0


In [173]:
# Column check
print("Initial columns:", df.columns.tolist())
df.columns = df.columns.str.strip()
print("Clean columns:", df.columns.tolist())

Initial columns: ['ID', ' Customer_Name', 'Order_ID', 'Order_Date', 'Product', ' Category', 'Quantity', 'Price', 'Payment_Method', 'Status', 'Total']
Clean columns: ['ID', 'Customer_Name', 'Order_ID', 'Order_Date', 'Product', 'Category', 'Quantity', 'Price', 'Payment_Method', 'Status', 'Total']


In [174]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 102 entries, 0 to 101
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              102 non-null    int64  
 1   Customer_Name   102 non-null    str    
 2   Order_ID        102 non-null    str    
 3   Order_Date      102 non-null    str    
 4   Product         102 non-null    str    
 5   Category        94 non-null     str    
 6   Quantity        97 non-null     str    
 7   Price           97 non-null     str    
 8   Payment_Method  102 non-null    str    
 9   Status          102 non-null    str    
 10  Total           88 non-null     float64
dtypes: float64(1), int64(1), str(9)
memory usage: 8.9 KB


Data type converions:
- `Order_Date` to datetime
- `Quantity` to int
- `Price` to float

Inter-column relations:
- `Customer_Name` = Customer_`ID` (but this is because the data is synthetic so we shouldn't be bothered here)
- $\text{Quantity} \times \text{Price} = \text{Total}$

In [175]:
df.isna().sum()

ID                 0
Customer_Name      0
Order_ID           0
Order_Date         0
Product            0
Category           8
Quantity           5
Price              5
Payment_Method     0
Status             0
Total             14
dtype: int64

## Data Fromatting and Type Conversion

### `Order_Date` to datetime.

In [176]:
df["Order_Date"].head(10)

0    11/22/2024
1      7/5/2025
2    12/23/2024
3     3/19/2025
4    10/20/2025
5    11/20/2024
6      2/2/2025
7      1/3/2025
8    10/23/2025
9      5/3/2025
Name: Order_Date, dtype: str

Does every value follow this `^\d{1,2}\/\d{1,2}\/\d{4}$` format (for example 11/22/2024)?

In [177]:
order_date_mask = df["Order_Date"].str.strip().str.match(r"^\d{1,2}\/\d{1,2}\/\d{4}$")

print("Non-matching values:", (~order_date_mask).sum())

display(df["Order_Date"][~order_date_mask])

Non-matching values: 2


14    Jan 5 2023
92           abc
Name: Order_Date, dtype: str

"Jan 5 2023" will automatically be converted to the majority format while "abc" will just be NaT.

In [178]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], format="%m/%d/%Y", errors="coerce")

Now every value is in YYYY-MM-DD format.

In [179]:
display(df["Order_Date"].head())

0   2024-11-22
1   2025-07-05
2   2024-12-23
3   2025-03-19
4   2025-10-20
Name: Order_Date, dtype: datetime64[us]

### `Quantity` to int

In [180]:
df["Quantity"].head()

0    3
1    2
2    1
3    5
4    1
Name: Quantity, dtype: str

Check if every value resembles a number `\d+`.

In [181]:
quantity_mask = df["Quantity"].str.strip().str.match(r"^\d+$")

print("Non-matching values:", (~quantity_mask).sum())

display(df["Quantity"][~quantity_mask])

Non-matching values: 8


6     NaN
17     -2
34     -5
64    NaN
67    NaN
92     4a
93    NaN
96    NaN
Name: Quantity, dtype: str

We convert all these 8 values to NaN.

In [182]:
def to_int(quantity_str: str):
    if pd.isna(quantity_str):
        return pd.NA

    s = quantity_str.strip()

    if s.isdigit():
        return int(s)

    return pd.NA

df["Quantity"] = df["Quantity"].apply(to_int).astype("Int64")

display(df["Quantity"].head())

0    3
1    2
2    1
3    5
4    1
Name: Quantity, dtype: Int64

### `Price` to float

In [183]:
df["Price"].head()

0        38
1       abd
2    389.05
3    233.92
4    552.51
Name: Price, dtype: str

Check for the values that resemble an actual number `^\d+\.{0,1}\d*$`.

In [184]:
price_mask = df["Price"].str.strip().str.match(r"^\d+\.{0,1}\d*$")

print("Non-matching values:", (~price_mask).sum())

display(df["Price"][~price_mask])

Non-matching values: 10


1              abd
10    four hundred
16             NaN
20            300$
24             NaN
30             NaN
37            -100
56             NaN
83             NaN
96             abd
Name: Price, dtype: str

Values like "four hundred", "300$", and "-100" are among the non-matching values. It's tempting whether to discard them as NaN or convert them to their most probably meant value. But since they're just 3 in number, it's better to discard them and later on interpolate them. However, I think it would be better to note down them along with their indices to check them after interpolation.

|Index|Price|
|---|---|
|10|four hundred|
|20|300$|
|37|-100|

In [185]:
def to_float(price_str):
    if pd.isna(price_str):
        return pd.NA
    
    s = price_str.strip()
    price_pattern = re.compile(r'^\d+\.{0,1}\d*$')  # 275 27.5 27.

    if price_pattern.fullmatch(s):
        return float(s)
    
    return pd.NA

df["Price"] = df["Price"].apply(to_float).astype("Float64")

display(df["Price"].head())

0      38.0
1      <NA>
2    389.05
3    233.92
4    552.51
Name: Price, dtype: Float64

### Formatting strings

In [186]:
str_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

print("String type columns:", str_cols)
display(df[str_cols].head())

String type columns: ['Customer_Name', 'Order_ID', 'Product', 'Category', 'Payment_Method', 'Status']


,Customer_Name,Order_ID,Product,Category,Payment_Method,Status
0,Customer_100,ORD-41285,Blender,Home,Cash on Delivery,Shipped
1,Customer_101,ORD-35783,Smartphone,Electronics,PayPal,Processing
2,Customer_102,ORD-84355,Tennis Racket,Sports,PayPal,Delivered
3,Customer_103,ORD-57811,Science,Books,PayPal,Processing
4,Customer_104,ORD-93614,Biography,Books,Cash on Delivery,Processing


In [187]:
# Trailing whitespace handling
for col in str_cols:
    df[col] = df[col].str.strip()

In [188]:
def format_masker(df: pd.DataFrame, col_format: list[tuple[str, str]], show=False):
    for col, format in col_format:
        mask = df[col].str.match(format)

        print(f"Non-matching values for column {col} and format {format}:", (~mask).sum())
        if show:
            display(df[col][(~mask)])
            print("-"*64)

col_format_pair = [("Customer_Name", r"^Customer_\d{3}$"), ("Order_ID", r"^ORD-\d{5}$")]
format_masker(df, col_format_pair)

Non-matching values for column Customer_Name and format ^Customer_\d{3}$: 0
Non-matching values for column Order_ID and format ^ORD-\d{5}$: 0


In [189]:
unique_method_cols = ["Product", "Category", "Payment_Method", "Status"]

for col in unique_method_cols:
    df[col] = df[col].str.title()
    print(f"{col}:", sorted(df[col].dropna().unique().tolist()))
    print("")

Product: ['Basketball', 'Biography', 'Blender', 'Comics', 'Fiction', 'Football', 'Headphones', 'Jacket', 'Jeans', 'Lamp', 'Laptop', 'Microwave', 'Science', 'Shoes', 'Smartphone', 'Smartwatch', 'T-Shirt', 'Tennis Racket', 'Vacuum', 'Yoga Mat']

Category: ['Books', 'Clothing', 'Electronic', 'Electronics', 'Home', 'Sports']

Payment_Method: ['Bank Transfer', 'Cash On Delivery', 'Credit Card', 'Paypal']

Status: ['Cancelled', 'Delivered', 'Processing', 'Returned', 'Shipped']



Every unique value is a valid value. Though, we need to make "Electronic" values plural in column Category to match with "Electronics" values.

In [190]:
df["Category"] = df["Category"].str.replace(r"^Electronic$", "Electronics", regex=True)

print("Category:", sorted(df["Category"].dropna().unique().tolist()))

Category: ['Books', 'Clothing', 'Electronics', 'Home', 'Sports']


### Conclusion

In [82]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 102 entries, 0 to 101
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID              102 non-null    int64         
 1   Customer_Name   102 non-null    str           
 2   Order_ID        102 non-null    str           
 3   Order_Date      100 non-null    datetime64[us]
 4   Product         102 non-null    str           
 5   Category        94 non-null     str           
 6   Quantity        94 non-null     Int64         
 7   Price           92 non-null     Float64       
 8   Payment_Method  102 non-null    str           
 9   Status          102 non-null    str           
 10  Total           88 non-null     float64       
dtypes: Float64(1), Int64(1), datetime64[us](1), float64(1), int64(1), str(6)
memory usage: 9.1 KB


In [83]:
df.isna().sum()

ID                 0
Customer_Name      0
Order_ID           0
Order_Date         2
Product            0
Category           8
Quantity           8
Price             10
Payment_Method     0
Status             0
Total             14
dtype: int64

A table showing the NA count change after data type conversion

| Column      | Old NA  | New NA    |
| ---         | ---     | ---       |
| `Order_Date`| 0       | 2         |
| `Quantity`  | 5       | 8         |
| `Price`     | 5       | 10        |
|             | **10**  | **20**    |












## ROMI Approach

### Relation

Track the rows where $\text{Quantity} \times \text{Price} \ne \text{Total}$

In [91]:
col_quantity = df["Quantity"]
col_price = df["Price"]
col_total = df["Total"]

ne_mask = (col_quantity * col_price != col_total)

print("Rows with broken relations:", ne_mask.sum())
display(df[ne_mask])

Rows with broken relations: 10


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
7,107,Customer_107,ORD-89122,2025-01-03,Biography,Books,5,587.68,PayPal,Returned,2938.400
15,115,Customer_115,ORD-72778,2025-01-10,Fiction,Books,2,796.84,Cash on Delivery,Returned,1115.576
42,142,Customer_142,ORD-69018,2025-10-30,Shoes,Clothing,5,645.26,Credit Card,Shipped,2258.410
51,151,Customer_151,ORD-56552,2025-07-01,Comics,Books,2,457.16,PayPal,Processing,640.024
60,160,Customer_160,ORD-96037,2025-01-03,Science,Books,5,242.52,Cash on Delivery,Returned,1212.600
73,173,Customer_173,ORD-42171,2025-10-11,Lamp,Home,5,351.89,Credit Card,Shipped,1759.450
79,179,Customer_179,ORD-99328,2025-09-03,Yoga Mat,Sports,3,868.67,PayPal,Delivered,2606.010
85,185,Customer_185,ORD-57123,2025-08-14,Football,Sports,2,730.92,Cash on Delivery,Cancelled,1023.288
99,199,Customer_199,ORD-82922,2025-01-22,Blender,Home,5,372.28,Credit Card,Shipped,1861.400
100,175,Customer_175,ORD-56651,2025-02-24,Headphones,Electronics,1,111.36,Credit Card,Processing,77.952


If `Total` is meant to be exactly `Quantity` × `Price`, then the inequality is worth investigating. It can point to data entry mistakes, rounding issues, or values that were coerced during cleaning. If `Total` includes tax, shipping, discounts, or refunds, then the inequality may be completely normal and you should not treat it as an error.

In our case, such clarifying information is not provided. So I choose to keep them as-is.

### Outlier

This **Outlier** step is for numeric values. We check for the existence of weird and inappropriate (extreme) values.

In [92]:
display(df.describe())

,ID,Order_Date,Quantity,Price,Total
count,102.000000,100,94.0,92.0,88.000000
mean,149.676471,2025-04-24 23:16:48,3.06383,619.737283,1230.927614
min,100.000000,2023-01-05 00:00:00,1.0,38.0,-20000.000000
25%,125.250000,2025-02-09 00:00:00,2.0,273.0,574.725000
50%,149.500000,2025-05-18 12:00:00,3.0,542.195,1127.180000
75%,174.750000,2025-08-04 00:00:00,5.0,737.94,2179.652500
max,199.000000,2025-11-06 00:00:00,5.0,10000.0,4722.700000
std,28.843779,NaN,1.52265,1022.448471,2620.332451


`Price`: 75% of the prices are within the range 273.0 and 737.94. Having values like 38.0 and 10,000.0 is suspicious, requiring further investigation.

`Total`: Having a minimum value of -20,000 is utterly preposterous.

In [115]:
Q1_price = df["Price"].quantile(0.25)
Q3_price = df["Price"].quantile(0.75)
IQR = Q3_price - Q1_price

print(f"Q1 = {Q1_price}")
print(f"Q3 = {Q3_price}")
print(f"IQR = {IQR:.2f}")

lower_limit = 100
upper_limit = Q3_price + IQR

lower_mask = df["Price"] < lower_limit
upper_mask = df["Price"] > upper_limit

Q1 = 273.0
Q3 = 737.94
IQR = 464.94


In [191]:
# Checking lower limit (prices less than 100)
display(df[lower_mask])

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,2024-11-22,Blender,Home,3,38.0,Cash On Delivery,Shipped,114.00
55,155,Customer_155,ORD-20182,2025-05-01,Shoes,Clothing,1,40.95,Paypal,Returned,40.95
77,177,Customer_177,ORD-64136,2025-07-28,Laptop,Electronics,1,69.48,Bank Transfer,Returned,69.48


It's hard to believe 3 blenders cost only 38.0 price units. Even a laptop costing 69.48 is ridicilous. So I dedcided to note the case down and move on. And honestly, it's really not part of my job or responsibility to indentifies such data meaninglessness. But if they are part of the outliers, then yes it's my job to handle and report those.

In [192]:
# Checking upper limit
display(df[upper_mask])

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
17,117,Customer_117,ORD-72751,2025-02-12,Blender,Electronics,<NA>,10000.0,Cash On Delivery,Processing,-20000.0


A price of 10,000.0 for blender (unknown quantity), relative to the rest of the dataset, is an obvious outlier. Even the total is -20,000.0. In addition, according to other blender sales, the category should be home, not electronics. So the question is, which option should I choose:

1. I drop the record
2. I make the nonsense entries NA
3. I drop the row after saving it in another dropped row accumulater dataframe

Four entries are invalid (nonsense), which accounts for the heart/essence of the record. In other words, this record is irreparably bad data. So option 2 is out of the options. Option 1 is data loss. If the client asks, "Why does the final dataset have one less row than the raw dataset?" we have nothing to show them. That's why we go with option 3. It even is the most professional way.

In [194]:
outlier_rows = pd.concat([df[lower_mask], df[upper_mask]])
display(outlier_rows)

rejected_records = pd.concat([rejected_records, outlier_rows])
display(rejected_records)

df.drop(outlier_rows.index, inplace=True)

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,2024-11-22,Blender,Home,3,38.0,Cash On Delivery,Shipped,114.00
55,155,Customer_155,ORD-20182,2025-05-01,Shoes,Clothing,1,40.95,Paypal,Returned,40.95
77,177,Customer_177,ORD-64136,2025-07-28,Laptop,Electronics,1,69.48,Bank Transfer,Returned,69.48
17,117,Customer_117,ORD-72751,2025-02-12,Blender,Electronics,<NA>,10000.0,Cash On Delivery,Processing,-20000.00


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,2024-11-22 00:00:00,Blender,Home,3,38.0,Cash On Delivery,Shipped,114.0
55,155,Customer_155,ORD-20182,2025-05-01 00:00:00,Shoes,Clothing,1,40.95,Paypal,Returned,40.95
77,177,Customer_177,ORD-64136,2025-07-28 00:00:00,Laptop,Electronics,1,69.48,Bank Transfer,Returned,69.48
17,117,Customer_117,ORD-72751,2025-02-12 00:00:00,Blender,Electronics,<NA>,10000.0,Cash On Delivery,Processing,-20000.0


### Mismatch

This **Mismatch** step is for strings (categorical values). In our case, we should check if, for example, every blender is categorized as a home utility and not electronics or something else.

For this specific dataset, we're going to match `Product` and `Category` columns.

In [204]:
# count how many unique categories each product has
category_counts = df.groupby("Product")["Category"].nunique()
display(category_counts)
print("-"*64)

problem_products = category_counts[category_counts > 1].index

if len(problem_products) > 0:
    print("Mismatched products found:")
    mismatched_rows = df[df["Product"].isin(problem_products)][["Product", "Category"]].drop_duplicates().dropna()
    display(mismatched_rows.sort_values(by="Product"))
else:
    print("No mismatches found. Every product belongs to exactly one category.")

Product
Basketball       2
Biography        1
Blender          1
Comics           1
Fiction          1
Football         1
Headphones       1
Jacket           1
Jeans            1
Lamp             2
Laptop           1
Microwave        2
Science          1
Shoes            1
Smartphone       1
Smartwatch       1
T-Shirt          2
Tennis Racket    1
Vacuum           2
Yoga Mat         2
Name: Category, dtype: int64

----------------------------------------------------------------
Mismatched products found:


,Product,Category
10,Basketball,Sports
46,Basketball,Electronics
28,Lamp,Electronics
31,Lamp,Home
16,Microwave,Home
74,Microwave,Electronics
34,T-Shirt,Clothing
66,T-Shirt,Electronics
23,Vacuum,Electronics
37,Vacuum,Home


In [208]:
product_category_map = {
    "Basketball": "Sports",
    "Lamp": "Home",
    "Microwave": "Home",
    "T-Shirt": "Clothing",
    "Vacuum": "Home",
    "Yoga Mat": "Sports"
}

df["Category"] = df["Product"].map(product_category_map).fillna(df["Category"])

# Optional: verify the fix worked by rerunning the check
category_counts = df.groupby("Product")["Category"].nunique()
problem_products = category_counts[category_counts > 1].index
print("Remaining mismatch products:", len(problem_products))

Remaining mismatch products: 0


### Interpolation

There is no interpolation for this dataset. Because, come on, this is a sales dataset. You can't just plug in best guess prices.